# WaSPS-DTW × CPAZMaL — Demo Notebook

This notebook demonstrates the full WaSPS-DTW classification pipeline on the
**CPAZMaL** SAR dataset (Sentinel-1 SAR time series of land-cover classes in the
French Alps, available on [HuggingFace](https://huggingface.co/datasets/musmb/CPAZMaL)).

Two classification modes are illustrated:

1. **K-Médoid with Soft-DTW Wasserstein** — per-class barycenters are estimated
   from training windows; each test window is assigned the label of its nearest
   barycenter (measured by Soft-DTW with W2 distance).

2. **Learning Shapelets with Soft-DTW Wasserstein** — a neural network learns
   discriminative subsequences (shapelets) of λ parameters. The distance between
   a time series and a shapelet is computed with Soft-DTW using the W2 cost
   between exponential distributions.

---
**Prerequisites**:
```bash
source venv/bin/activate      # from repo root
pip install h5py huggingface_hub torch tqdm
jupyter notebook notebooks/demo_cpazmal.ipynb
```

In [ ]:
import sys
from pathlib import Path

# Make sure src/ is on the Python path (run from repo root)
repo_root = Path(".").resolve()
src_dir = repo_root / "src"
sys.path.insert(0, str(src_dir))

import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

print(f"src/ on path: {src_dir}")

## 1 — Download & Load the CPAZMaL Dataset

The dataset is hosted on HuggingFace (`musmb/CPAZMaL`).  
Run the cell below once; subsequent runs will use the cached local copy.

In [ ]:
from dataloader.cpazmal_loader import MLDatasetLoader, download_cpazmal

# -----------------------------------------------------------------------
# Adjust DATA_DIR to wherever you want to store the HDF5 file.
# Set HF_TOKEN if the repo is private.
# -----------------------------------------------------------------------
DATA_DIR = Path("../data/cpazmal")   # relative to repo root
HF_TOKEN = None                      # replace with your HuggingFace token if needed

hdf5_files = list(DATA_DIR.glob("**/*.hdf5")) if DATA_DIR.exists() else []
if hdf5_files:
    HDF5_PATH = str(hdf5_files[0])
    print(f"Found cached HDF5: {HDF5_PATH}")
else:
    print("Downloading CPAZMaL from HuggingFace …")
    HDF5_PATH = download_cpazmal(str(DATA_DIR), token=HF_TOKEN)
    print(f"Downloaded to: {HDF5_PATH}")

In [ ]:
loader = MLDatasetLoader(HDF5_PATH)

summary = loader.get_statistics_summary()
print(f"Total groups : {summary['global']['n_groups']}")
print(f"Classes      : {loader.classes}")
print()
for cls, info in summary['by_class'].items():
    print(f"  {cls:25s}: {info['n_groups']:4d} groups")

## 2 — Extract Time Series

`extract_time_series` partitions the data into a **training** period and a
**prediction** period, extracts overlapping W×W spatial windows from each SAR
image stack, and reshapes each window `(W, W, T)` → `(T, W²)`.

Each row of the resulting array contains the W² pixel amplitudes observed at
time step *t*, which are treated as i.i.d. samples from an exponential
distribution with rate parameter λ_t.

In [ ]:
from dataloader.cpazmal_loader import extract_time_series

dataset = extract_time_series(
    loader,
    window_size=12,
    max_mask_value=1,
    max_mask_percentage=10.0,
    min_valid_percentage=50.0,
    orbit='DSC',
    polarization='HH',
    train_start='20200101',
    train_end='20201031',
    predict_start='20201101',
    predict_end='20201231',
    scale_type='amplitude',
    skip_optim_offset=False,
    verbose=True,
)

X_train    = dataset['X_train']       # (N,) object array — each element (T_train, W²)
X_predict  = dataset['X_predict']     # (N,) object array — each element (T_predict, W²)
y          = dataset['y']             # (N,) int — class labels
class_names = dataset['class_names']  # {int: str}

print(f"\nNumber of samples : {len(X_train)}")
print(f"Classes present   : {dict(Counter(class_names[i] for i in y))}")
print(f"Shape of X_train[0]: {X_train[0].shape}  (T_train, W²)")
print(f"Shape of X_predict[0]: {X_predict[0].shape}  (T_predict, W²)")

## 3 — Visualise Time Series per Class

We estimate the exponential rate parameter λ_t at each time step from the
W² pixel values (MLE: λ̂_t = 1 / mean(samples_t)), then plot the resulting
λ time series, coloured by land-cover class.

In [ ]:
from dataloader.classification_loader import estimate_parameters_for_samples

# Estimate λ for each training time series: (T_train, 1) per sample
params_train = [estimate_parameters_for_samples(xi) for xi in X_train]

print(f"λ parameter shape per sample: {params_train[0].shape}  (T_train, 1)")

In [ ]:
unique_classes = sorted(set(y))
palette = plt.cm.tab10.colors

fig, axes = plt.subplots(1, len(unique_classes), figsize=(5 * len(unique_classes), 4),
                          sharey=True)
if len(unique_classes) == 1:
    axes = [axes]

for ax, cls_id in zip(axes, unique_classes):
    cls_name = class_names[cls_id]
    indices = np.where(y == cls_id)[0]
    color = palette[cls_id % len(palette)]
    for idx in indices[:15]:   # show up to 15 examples
        ax.plot(params_train[idx].flatten(), color=color, alpha=0.4, linewidth=0.8)
    ax.set_title(cls_name)
    ax.set_xlabel("Time step")
    ax.grid(alpha=0.3)

axes[0].set_ylabel("λ (rate parameter)")
fig.suptitle("Exponential rate λ time series — training period", fontsize=13)
plt.tight_layout()
plt.show()

## 4 — K-Médoid Classification via Soft-DTW Wasserstein Barycenters

### Strategy
1. For each class, compute a **Soft-DTW barycenter** of the λ time series in
   the training set.  The barycenter optimisation is performed with SGD and a
   Wasserstein-2 distance between exponential distributions.
2. Each test (prediction) sample is assigned the label of the **nearest
   barycenter** under the Soft-DTW Wasserstein distance.

> Core functions `compute_barycenter_wasserstein_sgd` and
> `compute_sdtw_distance_wasserstein` are in `src/sdtw/classification_methods.py`.

In [ ]:
from sdtw.classification_methods import (
    compute_barycenter_wasserstein_sgd,
    compute_sdtw_distance_wasserstein,
)

# Tune these for a real experiment; kept small here for speed.
GAMMA = 0.5
SGD_EPOCHS = 50
SGD_LR = 0.05

# Group training params by class
params_by_class = {cls_id: [] for cls_id in unique_classes}
for i, lam in enumerate(params_train):
    params_by_class[y[i]].append(lam)

for cls_id in unique_classes:
    print(f"  {class_names[cls_id]:25s}: {len(params_by_class[cls_id])} training samples")

In [ ]:
print("Computing Soft-DTW Wasserstein barycenters (may take a few minutes)…")
barycenters = {}
for cls_id in unique_classes:
    samples = params_by_class[cls_id]
    print(f"  Computing barycenter for '{class_names[cls_id]}' ({len(samples)} samples) …",
          flush=True)
    bary = compute_barycenter_wasserstein_sgd(
        samples,
        gamma=GAMMA,
        learning_rate=SGD_LR,
        num_epochs=SGD_EPOCHS,
        batch_size=4,
        verbose=False,
    )
    barycenters[cls_id] = bary  # (T_train, 1)

print("Done.")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
for cls_id, bary in barycenters.items():
    ax.plot(bary.flatten(), label=class_names[cls_id],
            color=palette[cls_id % len(palette)], linewidth=2)
ax.set_title("Soft-DTW Wasserstein barycenters per class (training set)")
ax.set_xlabel("Time step")
ax.set_ylabel("λ (rate parameter)")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Classify prediction set — assign nearest barycenter
print("Classifying prediction samples …")
y_pred_kmedoid = []
for xi in X_predict:
    lam_i = estimate_parameters_for_samples(xi)   # (T_predict, 1)
    distances = {}
    for cls_id, bary in barycenters.items():
        min_T = min(lam_i.shape[0], bary.shape[0])
        dist = compute_sdtw_distance_wasserstein(
            lam_i[:min_T], bary[:min_T], gamma=GAMMA
        )
        distances[cls_id] = dist
    y_pred_kmedoid.append(min(distances, key=distances.get))

y_pred_kmedoid = np.array(y_pred_kmedoid)
acc_kmedoid = np.mean(y_pred_kmedoid == y)
print(f"\nK-Médoid accuracy: {acc_kmedoid:.3f}  ({100*acc_kmedoid:.1f} %)")

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

labels = [class_names[i] for i in unique_classes]
cm = confusion_matrix(y, y_pred_kmedoid, labels=unique_classes)
disp = ConfusionMatrixDisplay(cm, display_labels=labels)
fig, ax = plt.subplots(figsize=(7, 6))
disp.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title("K-Médoid — confusion matrix (prediction period)")
plt.tight_layout()
plt.show()

## 5 — Learning Shapelets with Soft-DTW Wasserstein

### How it works
Each shapelet is a short sequence of learnable λ parameters (stored in log
space so that λ > 0 is always enforced).  
The distance between a time series and a shapelet is the **minimum
Soft-DTW Wasserstein distance** over all sliding-window positions.

A linear layer maps the shapelet-distance feature vector to class logits.
The model (shapelets + linear) is trained end-to-end with cross-entropy loss.

### Input format
The model expects `(batch, 1, T)` tensors of λ values.  We stack the
estimated parameters by truncating to the shortest time series.

In [ ]:
import torch
from optimizer.learning_shapelets import LearningShapelets

# Build (N, 1, T_min) λ matrix
T_min = min(p.shape[0] for p in params_train)
print(f"Shortest training series: T_min = {T_min}")

X_lambda = np.stack([p[:T_min, 0] for p in params_train], axis=0)  # (N, T_min)
X_lambda = X_lambda[:, np.newaxis, :]                               # (N, 1, T_min)
X_lambda = np.clip(X_lambda, 1e-6, None)                           # ensure λ > 0

# Re-map labels to 0-based
unique_labels = sorted(set(y))
label_map = {orig: new for new, orig in enumerate(unique_labels)}
Y_labels_0 = np.array([label_map[yi] for yi in y])

num_classes = len(unique_labels)
print(f"X_lambda shape: {X_lambda.shape}")
print(f"Number of classes: {num_classes}")

In [ ]:
# Shapelet lengths and counts — keep small for the demo
SHAPELET_CONFIG = {6: 4, 10: 4}   # {length: num_shapelets}
GAMMA_SHAPELET = 0.5
TO_CUDA = torch.cuda.is_available()

clf = LearningShapelets(
    shapelets_size_and_len=SHAPELET_CONFIG,
    loss_func=torch.nn.CrossEntropyLoss(),
    in_channels=1,
    num_classes=num_classes,
    dist_measure='soft_dtw_wasserstein',
    gamma=GAMMA_SHAPELET,
    to_cuda=TO_CUDA,
    verbose=1,
)
clf.set_optimizer(torch.optim.Adam(clf.model.parameters(), lr=1e-2))

print("Model created.")
print(f"  Shapelet config: {SHAPELET_CONFIG}")
print(f"  Device: {'CUDA' if TO_CUDA else 'CPU'}")

In [ ]:
# NOTE: The SoftDTW-Wasserstein forward is a sequential Python loop
#   (batch × shapelets × windows).  Use small batch sizes on CPU.
EPOCHS = 20
BATCH_SIZE = 8

print(f"Training for {EPOCHS} epochs (batch_size={BATCH_SIZE}) …")
losses = clf.fit(
    X_lambda,
    Y_labels_0,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    shuffle=True,
)

plt.figure(figsize=(7, 3))
plt.plot(losses, linewidth=1.2)
plt.title("Learning Shapelets — training loss")
plt.xlabel("Update step")
plt.ylabel("Cross-entropy loss")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Evaluate on the training set as a sanity check
logits = clf.predict(X_lambda, batch_size=BATCH_SIZE)  # (N, C)
y_pred_shapelet = logits.argmax(axis=1)

acc_shap = np.mean(y_pred_shapelet == Y_labels_0)
print(f"Learning Shapelets train accuracy: {acc_shap:.3f}  ({100*acc_shap:.1f} %)")

In [ ]:
# Visualise learned shapelets
all_shapelets = clf.get_shapelets()  # (num_total, 1, max_len) — NaN-padded
num_total = all_shapelets.shape[0]

fig, axes = plt.subplots(1, num_total, figsize=(3.5 * num_total, 3), sharey=False)
if num_total == 1:
    axes = [axes]
for ax, shap in zip(axes, all_shapelets):
    s_valid = shap[0, ~np.isnan(shap[0, :])]
    ax.plot(s_valid, marker='o', markersize=4)
    ax.set_title(f"len={len(s_valid)}")
    ax.set_xlabel("Shapelet position")
    ax.grid(alpha=0.3)
axes[0].set_ylabel("λ")
fig.suptitle("Learned Soft-DTW Wasserstein shapelets", fontsize=13)
plt.tight_layout()
plt.show()

## 6 — Comparison Summary

In [ ]:
print("="*55)
print("RESULTS SUMMARY")
print("="*55)
print(f"Dataset         : CPAZMaL (DSC HH, amplitude)")
print(f"Window size     : 12×12 pixels")
print(f"Samples         : {len(X_train)} (train)  /  {len(X_predict)} (predict)")
print(f"Classes         : {[class_names[c] for c in unique_classes]}")
print(f"γ (SoftDTW)     : {GAMMA}")
print()
print(f"K-Médoid acc.          : {acc_kmedoid:.3f}  ({100*acc_kmedoid:.1f} %)")
print(f"Learning Shapelets acc.: {acc_shap:.3f}  ({100*acc_shap:.1f} %)  "
      f"[training set, {EPOCHS} epochs]")
print()
print("Note: Learning Shapelets accuracy on *training* set is shown.")
print("For a fair comparison, evaluate both methods on a held-out test set.")